**The Scenario:** You act as a university data analyst. The legacy admissions portal crashed during a system migration, and the backup applicant data is a mess. You must clean the "Applicant Records" dataset.
- Handling Missing Data: The dataset is riddled with NaNs. You must use .dropna() to remove rows completely missing an Application ID. You will use .fillna() to replace missing Intended Major and High School State values with "Unknown" that were lost during the export.
- Handling Duplicates: A server glitch caused applicants to accidentally submit identical applications twice. You must use .duplicated() to find identical application rows and .drop_duplicates(keep='first') to remove the errors and keep only the original submission.

In [16]:
import pandas as pd

df = pd.read_csv("admissions_messy.csv")
df.info()
df.head()

df = df.dropna(subset=["Application ID"])

df["Intended Major"] = df["Intended Major"].fillna("Unknown")
df["High School State"] = df["High School State"].fillna("Unknown")
# print(df)
duplicates = df[df.duplicated()]
print(f"Duplicate rows:\n {duplicates}")

df = df.drop_duplicates(keep="first")
df.info()
df.head()
df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1050 entries, 0 to 1049
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Application ID     943 non-null    float64
 1   Applicant Name     1050 non-null   object 
 2   Email              1050 non-null   object 
 3   Intended Major     898 non-null    object 
 4   High School State  874 non-null    object 
 5   Application Fee    1050 non-null   object 
 6   Submission Date    1050 non-null   object 
dtypes: float64(1), object(6)
memory usage: 57.6+ KB
Duplicate rows:
       Application ID       Applicant Name                  Email  \
77            1210.0        Bruce Allen        ballen@skynet.com   
144           1986.0          ALICE Doe           adoe@stark.com   
147           1319.0         peter DOE              pdoe@edu.org   
160           1513.0           ARTHUR Doe         adoe@gmail.com   
177           1059.0      Victor CONNOR        vconn

,Application ID,Applicant Name,Email,Intended Major,High School State,Application Fee,Submission Date
0,1352.0,wanda DOE,wdoe@gmail.com,Math,NV,$297.91,pending
1,1689.0,PETER JONES,pjones@edu.org,Journalism,Unknown,"$1,939.68",08/14/2023
2,1485.0,wanda MAXIMOFF,wmaximoff@yahoo.com,Math,Unknown,$277.90,2023/06/06
3,1388.0,PETER Parker,pparker@yahoo.com,Agriculture,AZ,$370.25,2023-02-24
4,1031.0,bruce rogers,brogers@stark.com,Physics,NV,$53.93,"Oct 16, 2023"
...,...,...,...,...,...,...,...
1043,1871.0,Arthur Wayne,awayne@gmail.com,Art,NJ,"$1,619.10",05-07-2023
1044,1087.0,John Stone,jstone@gmail.com,Computer Science,KS,"$1,582.76",2023/05/27
1045,1330.0,diana MAXIMOFF,dmaximoff@edu.org,Unknown,AZ,$862.30,10-18-2023
1046,1466.0,STEVE Allen,sallen@edu.org,Journalism,Unknown,$389.22,10-20-2023


**Data Type Conversions:** 
- Application Fee is imported as a string (e.g., "$150.50"). You must remove the symbols and use pd.to_numeric() to turn it into a float.
- Replace “pending” and “unknown” text entries in the dates with the previous valid datetime (forward filling).
- Convert Submission Date from messy text into a proper datetime object using pd.to_datetime().
- Convert the Submission Date format from various mixed formats into a uniform "M/D/Y" format.

In [19]:
df["Application Fee"] = (df["Application Fee"].replace(r"[\$,]", "", regex=True))
df["Application Fee"] = pd.to_numeric(df["Application Fee"], errors="coerce")

df["Submission Date"] = df["Submission Date"].replace(["pending", "unknown"], pd.NA)
df["Submission Date"] = df["Submission Date"].fillna(method="ffill")

df["Submission Date"] = pd.to_datetime(df["Submission Date"], errors="coerce")

# df["Submission Date"] = df["Submission Date"].dt.strftime("%-m/%-d/%Y")
df["Submission Date"] = df["Submission Date"].dt.strftime("%m/%d/%Y")

print(df[["Application Fee", "Submission Date"]].head())
df.info()

   Application Fee Submission Date
0           297.91             NaN
1          1939.68      08/14/2023
2           277.90      08/14/2023
3           370.25      08/14/2023
4            53.93      08/14/2023
<class 'pandas.core.frame.DataFrame'>
Index: 898 entries, 0 to 1047
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Application ID     898 non-null    float64
 1   Applicant Name     898 non-null    object 
 2   Email              898 non-null    object 
 3   Intended Major     898 non-null    object 
 4   High School State  898 non-null    object 
 5   Application Fee    898 non-null    float64
 6   Submission Date    897 non-null    object 
dtypes: float64(2), object(5)
memory usage: 56.1+ KB


C:\Users\Dell\AppData\Local\Temp\ipykernel_15440\1546740097.py:5: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["Submission Date"] = df["Submission Date"].fillna(method="ffill")


 **String Manipulation:** The Applicant Name column has chaotic capitalization and extra spaces. You will use .str.title() and .str.strip() to fix this. You must also extract the email domain (e.g., extracting "yahoo.com" from "student@yahoo.com") using .str.split() to see which email providers are most common.

In [12]:
df["Applicant Name"] = (df["Applicant Name"].str.strip().str.title())
df["Applicant Name"] = (df["Applicant Name"].str.strip().str.replace(r"\s+", " ", regex=True).str.title())
print(df["Applicant Name"])

df["Email Domain"] = df["Email"].str.split("@").str[-1]
domain_counts = df["Email Domain"].value_counts()

print(domain_counts.head())

top_provider = df["Email Domain"].mode()[0]
print("Most common email provider:", top_provider)

df.to_csv("cleaned_applicant_records.csv", index=False)

0            Wanda Doe
1          Peter Jones
2       Wanda Maximoff
3         Peter Parker
4         Bruce Rogers
             ...      
1045    Diana Maximoff
1046       Steve Allen
1047    Charlie Rogers
1048       Peter Wayne
1049     Charlie Brown
Name: Applicant Name, Length: 1050, dtype: object
Email Domain
gmail.com         167
stark.com         162
yahoo.com         150
dailybugle.com    147
edu.org           145
Name: count, dtype: int64
Most common email provider: gmail.com


In [9]:
df = pd.read_csv("admissions_messy.csv")
df

,Application ID,Applicant Name,Email,Intended Major,High School State,Application Fee,Submission Date
0,1352.0,wanda DOE,wdoe@gmail.com,Math,NV,$297.91,pending
1,1689.0,PETER JONES,pjones@edu.org,Journalism,NaN,"$1,939.68",08/14/2023
2,1485.0,wanda MAXIMOFF,wmaximoff@yahoo.com,Math,NaN,$277.90,2023/06/06
3,1388.0,PETER Parker,pparker@yahoo.com,Agriculture,AZ,$370.25,2023-02-24
4,1031.0,bruce rogers,brogers@stark.com,Physics,NV,$53.93,"Oct 16, 2023"
...,...,...,...,...,...,...,...
1045,1330.0,diana MAXIMOFF,dmaximoff@edu.org,NaN,AZ,$862.30,10-18-2023
1046,1466.0,STEVE Allen,sallen@edu.org,Journalism,NaN,$389.22,10-20-2023
1047,1121.0,Charlie ROGERS,crogers@stark.com,History,AZ,"$1,746.27","Aug 17, 2023"
1048,1299.0,peter Wayne,pwayne@dailybugle.com,Journalism,KS,$689.05,2023-06-06
